# 🧠 ResumeIQ AI — Module 1: Resume Category Classifier

**Problem:** Given a raw resume text, classify it into one of 24 job categories.

**Dataset:** [Kaggle — Resume Dataset](https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset) by Sneha Anbhawal  
- 2,400+ labelled resumes across 24 job categories
- Columns: `ID`, `Resume_str`, `Resume_html`, `Category`

**Categories:** HR · Designer · Information-Technology · Teacher · Advocate · Business-Development ·  
Healthcare · Fitness · Agriculture · BPO · Sales · Consultant · Digital-Media · Automobile ·  
Chef · Finance · Apparel · Engineering · Accountant · Construction · Public-Relations · Banking · Arts · Aviation

**Models Trained & Compared:**
- Logistic Regression (baseline)
- Random Forest
- Linear SVC
- XGBoost
- Best model selected by 5-fold stratified cross-validation F1 (macro)

**Metrics:** Accuracy · Precision · Recall · F1 (macro) · Confusion Matrix · Per-class F1

In [ ]:
# Install dependencies (run once)
# !pip install pandas scikit-learn xgboost matplotlib seaborn joblib

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import re
import pickle
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score
)
import xgboost as xgb

os.makedirs('plots', exist_ok=True)
os.makedirs('../data/models', exist_ok=True)

print('Libraries loaded ✓')
print(f'pandas {pd.__version__} | xgboost {xgb.__version__}')

## Step 1 — Download & Load Dataset

**Dataset:** https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset

```bash
# Option A — Kaggle CLI (recommended)
kaggle datasets download -d snehaanbhawal/resume-dataset -p training/data/ --unzip

# Option B — Manual
# Go to the Kaggle link above → Download → extract Resume.csv into training/data/
```

The downloaded file is named **`Resume.csv`** with columns: `ID`, `Resume_str`, `Resume_html`, `Category`

In [ ]:
# ── Auto-detect dataset file and text column ──────────────────────────────────
POSSIBLE_FILES = [
    ('data/Resume.csv',                'Resume_str'),   # snehaanbhawal dataset
    ('data/UpdatedResumeDataSet.csv',  'Resume'),       # jillanisofttech dataset
    ('data/resume_dataset.csv',        'Resume_str'),
    ('data/resume.csv',                'Resume_str'),
]

df = None
TEXT_COL = None

for path, col in POSSIBLE_FILES:
    if os.path.exists(path):
        df = pd.read_csv(path)
        # Auto-detect text column in case it differs
        if col in df.columns:
            TEXT_COL = col
        elif 'Resume_str' in df.columns:
            TEXT_COL = 'Resume_str'
        elif 'Resume' in df.columns:
            TEXT_COL = 'Resume'
        else:
            # Use the column with the longest average string length
            str_cols = df.select_dtypes('object').columns
            TEXT_COL = max(str_cols, key=lambda c: df[c].str.len().mean())
        print(f'Loaded: {path}')
        print(f'Text column detected: "{TEXT_COL}"')
        break

if df is None:
    print('Dataset not found. Attempting Kaggle API download...')
    os.system('kaggle datasets download -d snehaanbhawal/resume-dataset -p data/ --unzip')
    df = pd.read_csv('data/Resume.csv')
    TEXT_COL = 'Resume_str'

print(f'\nShape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print(f'\nSample text (first 200 chars):\n{str(df[TEXT_COL].iloc[0])[:200]}')

## Step 2 — Exploratory Data Analysis (EDA)

In [ ]:
print('Shape:', df.shape)
print(f'\nNull values in text column: {df[TEXT_COL].isnull().sum()}')
print(f'Null values in Category:    {df["Category"].isnull().sum()}')
print(f'\nNumber of categories: {df["Category"].nunique()}')
print('\nClass distribution:')
print(df['Category'].value_counts().to_string())

In [ ]:
# Category distribution bar chart
counts = df['Category'].value_counts()
fig, ax = plt.subplots(figsize=(14, 7))
colors = plt.cm.tab20(np.linspace(0, 1, len(counts)))
bars = ax.barh(counts.index, counts.values, color=colors)
ax.set_xlabel('Number of Resumes', fontsize=12)
ax.set_title('Resume Category Distribution (Kaggle Dataset)', fontsize=14, fontweight='bold')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)
plt.tight_layout()
plt.savefig('plots/01_category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total categories: {df["Category"].nunique()}')

In [ ]:
# Resume text length analysis
df['text_length'] = df[TEXT_COL].apply(lambda x: len(str(x).split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['text_length'], bins=50, color='#3b82f6', edgecolor='white')
axes[0].set_title('Resume Word Count Distribution')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df['text_length'].median(), color='red', linestyle='--',
                label=f'Median: {df["text_length"].median():.0f}')
axes[0].legend()

category_lengths = df.groupby('Category')['text_length'].median().sort_values()
axes[1].barh(category_lengths.index, category_lengths.values, color='#8b5cf6')
axes[1].set_title('Median Word Count by Category')
axes[1].set_xlabel('Median Word Count')

plt.tight_layout()
plt.savefig('plots/01_text_length.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean word count:   {df['text_length'].mean():.0f}")
print(f"Median word count: {df['text_length'].median():.0f}")
print(f"Max word count:    {df['text_length'].max()}")
print(f"Min word count:    {df['text_length'].min()}")

## Step 3 — Text Preprocessing

In [ ]:
def clean_resume_text(text: str) -> str:
    """Clean resume text: remove noise, normalise whitespace."""
    text = str(text)
    text = re.sub(r'http\S+|www\S+', ' ', text)          # URLs
    text = re.sub(r'\S+@\S+', ' ', text)                  # emails
    text = re.sub(r'[\+\(]?[\d\s\-\(\)]{7,15}', ' ', text) # phones
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)           # special chars
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

df['cleaned_resume'] = df[TEXT_COL].apply(clean_resume_text)

# Drop rows where cleaning resulted in empty text
df = df[df['cleaned_resume'].str.len() > 50].reset_index(drop=True)
print(f'Rows after cleaning: {len(df)}')

print('\n--- BEFORE ---')
print(str(df[TEXT_COL].iloc[0])[:300])
print('\n--- AFTER ---')
print(df['cleaned_resume'].iloc[0][:300])

## Step 4 — Feature Engineering: TF-IDF Vectorization

In [ ]:
# Encode labels
le = LabelEncoder()
y = le.fit_transform(df['Category'])
X_text = df['cleaned_resume']

print(f'Number of classes: {len(le.classes_)}')
print(f'Classes: {list(le.classes_)}')

# Train/test split — stratified so every category is represented
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.20, random_state=42, stratify=y
)
print(f'\nTrain size: {len(X_train_text)} | Test size: {len(X_test_text)}')

# TF-IDF — fit ONLY on train to prevent data leakage
tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),   # unigrams + bigrams
    sublinear_tf=True,    # log(1 + tf) scaling reduces high-freq term dominance
    min_df=2,             # ignore terms appearing in < 2 docs
    stop_words='english',
)
X_train = tfidf.fit_transform(X_train_text)
X_test  = tfidf.transform(X_test_text)

print(f'\nTF-IDF train matrix: {X_train.shape}')
print(f'TF-IDF test matrix:  {X_test.shape}')
print(f'Vocabulary size:     {len(tfidf.vocabulary_)}')

## Step 5 — Train & Compare 4 Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, C=1.0, solver='liblinear', multi_class='ovr'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    ),
    'Linear SVC': CalibratedClassifierCV(
        LinearSVC(max_iter=2000, C=1.0)
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        eval_metric='mlogloss', random_state=42, n_jobs=-1,
        use_label_encoder=False,
    ),
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average='macro')
    cv_scores = cross_val_score(model, X_train, y_train,
                                cv=cv, scoring='f1_macro', n_jobs=-1)
    results[name] = {
        'model': model, 'accuracy': acc, 'f1_macro': f1,
        'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(),
        'y_pred': y_pred,
    }
    print(f'  Accuracy: {acc:.4f} | F1 Macro: {f1:.4f} | CV F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

print('\nAll models trained ✓')

In [ ]:
# Model comparison table + chart
comparison_df = pd.DataFrame([
    {'Model': name, 'Test Accuracy': v['accuracy'],
     'F1 Macro': v['f1_macro'], 'CV F1 Mean': v['cv_mean']}
    for name, v in results.items()
]).set_index('Model')

print('=== Model Comparison ===')
print(comparison_df.round(4).to_string())

ax = comparison_df[['Test Accuracy', 'F1 Macro', 'CV F1 Mean']].plot(
    kind='bar', figsize=(10, 5), rot=15, colormap='Set2'
)
ax.set_title('Model Comparison — Resume Category Classifier', fontsize=14, fontweight='bold')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.legend(loc='lower right')
ax.axhline(0.9, color='red', linestyle='--', alpha=0.4, label='90%')
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=8)
plt.tight_layout()
plt.savefig('plots/01_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 6 — Best Model: Detailed Evaluation

In [ ]:
best_name = max(results, key=lambda k: results[k]['cv_mean'])
best = results[best_name]

print(f'Best model: {best_name}')
print(f'  Test Accuracy  : {best["accuracy"]:.4f}  ({best["accuracy"]*100:.2f}%)')
print(f'  F1 Macro       : {best["f1_macro"]:.4f}')
print(f'  CV F1 (5-fold) : {best["cv_mean"]:.4f} ± {best["cv_std"]:.4f}')
print()
print('=== Classification Report ===')
print(classification_report(
    y_test, best['y_pred'],
    target_names=le.classes_,
    digits=4
))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, best['y_pred'])
fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=le.classes_, yticklabels=le.classes_, ax=ax
)
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plots/01_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-class F1 sorted bar chart
report = classification_report(
    y_test, best['y_pred'], target_names=le.classes_, output_dict=True
)
class_f1 = {cls: report[cls]['f1-score'] for cls in le.classes_}
sorted_f1 = dict(sorted(class_f1.items(), key=lambda x: x[1]))

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#ef4444' if v < 0.8 else '#f59e0b' if v < 0.9 else '#22c55e'
          for v in sorted_f1.values()]
bars = ax.barh(list(sorted_f1.keys()), list(sorted_f1.values()), color=colors)
ax.set_xlabel('F1 Score')
ax.set_title(f'Per-Class F1 Score — {best_name}', fontsize=13, fontweight='bold')
ax.axvline(0.9, color='navy', linestyle='--', alpha=0.5, label='0.90 threshold')
ax.legend()
for bar, val in zip(bars, sorted_f1.values()):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('plots/01_per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7 — Top TF-IDF Discriminating Terms per Category

In [ ]:
# Use Logistic Regression coefficients to find top discriminating terms
lr_model = results['Logistic Regression']['model']
feature_names = tfidf.get_feature_names_out()

print('Top 8 discriminating terms per category:')
print('=' * 70)
for i, class_name in enumerate(le.classes_):
    top_idx = lr_model.coef_[i].argsort()[-8:][::-1]
    top_terms = [feature_names[j] for j in top_idx]
    print(f'{class_name:25s}: {" | ".join(top_terms)}')

## Step 8 — Save Best Model

In [ ]:
bundle = {
    'model':         best['model'],
    'tfidf':         tfidf,
    'label_encoder': le,
    'model_name':    best_name,
    'accuracy':      float(best['accuracy']),
    'f1_macro':      float(best['f1_macro']),
    'cv_mean':       float(best['cv_mean']),
    'n_classes':     len(le.classes_),
    'classes':       list(le.classes_),
}

with open('../data/models/resume_classifier.pkl', 'wb') as f:
    pickle.dump(bundle, f)

print('Model saved → data/models/resume_classifier.pkl')
print(f'Model      : {best_name}')
print(f'Accuracy   : {best["accuracy"]*100:.2f}%')
print(f'F1 Macro   : {best["f1_macro"]*100:.2f}%')
print(f'CV F1 Mean : {best["cv_mean"]*100:.2f}%')

## Results Summary

| Step | Detail |
|---|---|
| **Dataset** | Kaggle Resume Dataset — snehaanbhawal |
| **Dataset URL** | https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset |
| **Size** | 2,400+ resumes, 24 job categories |
| **Preprocessing** | URL/email/phone removal → lowercase → TF-IDF (15k features, 1-2 ngrams, sublinear_tf) |
| **Models Compared** | Logistic Regression, Random Forest, Linear SVC, XGBoost |
| **Selection Criterion** | 5-fold Stratified Cross-Validation F1 Macro |
| **Train / Test Split** | 80% / 20% stratified |

**Role in ResumeIQ AI:**  
This classifier powers **Engine 1 (Resume Intelligence Engine)** — it predicts the candidate's target job role directly from resume text, enabling the system to contextualise skill matching and hiring recommendations.